# Gold: estrella academica

Ejecuta `sql/gold/university.sql`: `dim_student`, `dim_professor`, `dim_course`, `dim_semester` + `fact_enrollment` (grano=inscripcion, con rollup de notas) y `fact_grade` (grano=calificacion individual).

**Requiere que `01_dim_date.ipynb` ya haya corrido** (las FKs de fecha dependen de `gold.dim_date`).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from pathlib import Path
from utils.db import get_psycopg2_connection, get_engine

engine = get_engine()
SQL_GOLD = Path("/home/jovyan/work/sql/gold")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

In [2]:
run_sql_file(SQL_GOLD / "university.sql")

OK: university.sql ejecutado


## 1. Conteos: gold vs. silver (deben coincidir exacto)

In [3]:
pd.read_sql("""
    SELECT 'dim_student' t, (SELECT count(*) FROM silver.university__students) silver, (SELECT count(*) FROM gold.dim_student) gold
    UNION ALL SELECT 'dim_professor', (SELECT count(*) FROM silver.university__professors), (SELECT count(*) FROM gold.dim_professor)
    UNION ALL SELECT 'dim_course', (SELECT count(*) FROM silver.university__courses), (SELECT count(*) FROM gold.dim_course)
    UNION ALL SELECT 'dim_semester', (SELECT count(*) FROM silver.university__semesters), (SELECT count(*) FROM gold.dim_semester)
    UNION ALL SELECT 'fact_enrollment', (SELECT count(*) FROM silver.university__enrollments), (SELECT count(*) FROM gold.fact_enrollment)
    UNION ALL SELECT 'fact_grade', (SELECT count(*) FROM silver.university__grades), (SELECT count(*) FROM gold.fact_grade)
""", engine)

,t,silver,gold
0,dim_student,5000,5000
1,dim_professor,200,200
2,dim_course,300,300
3,dim_semester,8,8
4,fact_enrollment,25000,25000
5,fact_grade,60000,60000


## 2. Pregunta de negocio: rendimiento promedio por departamento de curso

(usa `dim_course.department`, no el del profesor -- son independientes, ver `docs/calidad_datos.md`)

In [4]:
pd.read_sql("""
    SELECT c.department,
           count(*) AS inscripciones,
           round(avg(f.avg_score), 1) AS promedio,
           round(100.0 * avg(f.is_passing::int), 1) AS pct_aprobados
    FROM gold.fact_enrollment f
    JOIN gold.dim_course c ON c.course_id = f.course_id
    WHERE f.avg_score IS NOT NULL
    GROUP BY c.department
    ORDER BY promedio DESC
""", engine)

,department,inscripciones,promedio,pct_aprobados
0,cs,3662,75.1,95.7
1,biology,3411,75.1,95.9
2,math,1765,75.0,95.5
3,chemistry,2505,75.0,95.3
4,history,2378,74.8,95.5
5,economics,2612,74.8,95.6
6,literature,2547,74.8,94.9
7,physics,3906,74.7,95.0


## 3. Pregunta de negocio: deserción (`status = dropped`) por semestre

In [5]:
pd.read_sql("""
    SELECT s.code AS semestre,
           count(*) AS inscripciones,
           count(*) FILTER (WHERE f.status = 'dropped') AS abandonos,
           round(100.0 * count(*) FILTER (WHERE f.status = 'dropped') / count(*), 1) AS pct_abandono
    FROM gold.fact_enrollment f
    JOIN gold.dim_semester s ON s.semester_id = f.semester_id
    GROUP BY s.code
    ORDER BY s.code
""", engine)

,semestre,inscripciones,abandonos,pct_abandono
0,2022-1,3164,326,10.3
1,2022-2,3199,318,9.9
2,2023-1,3143,330,10.5
3,2023-2,3014,300,10.0
4,2024-1,3076,298,9.7
5,2024-2,3108,301,9.7
6,2025-1,3110,319,10.3
7,2025-2,3186,311,9.8
